# L02 · 声音的数字表示——采样（sampling）、数组、第一个可听正弦波（sine wave）

**学习目标**
1. 理解采样：连续声音 → numpy 数组 → 可再生的声音
2. 实现 `samples_count`：N = duration × sample_rate
3. 实现 `make_time_axis`：生成时间轴
4. 理解数组形状（shape）：1D 信号 vs 2D 矩阵的区别
5. 实现 `make_sine`：生成正弦波并播放
6. 实现 `signal_summary`：汇总信号统计量

> 这四个函数是整个课程的「原子操作」——后续所有 notebook 都直接或间接调用它们。

## 本课学习方法：先手算，再让代码验证

Aurora 课程的核心方法不是「运行看输出」，而是：

```
① 拿到公式，先在纸上或 Markdown 里推导
② 用极小的例子（例如 duration=0.5s, sr=8Hz → N=4）手算结果
③ 写代码实现
④ 运行 assert，让代码和手算结果对答案
```

**为什么这样做？**  
面试中没有终端，只有白板。「先手算」的练习
直接训练你在无代码环境中推导结果的能力。

本课每个任务都会先给一张推导表，等你填完再写代码。

## 1. 声音的物理本质

声音是空气**压强随时间的变化**。麦克风把压强转化为电压，
模数转换器（ADC）每隔固定时间测一次电压值，存成数字：

```
连续压强波形
  ╭──╮     ╭──╮
──╯  ╰──╮──╯  ╰──  ···
         ╰──╯
        ↓ 每 1/sr 秒采一次样（ADC）
[0.0, 0.6, 1.0, 0.6, 0.0, -0.6, -1.0, -0.6, ···]   ← numpy float64 数组
```

**采样率（sample rate, sr）**：每秒采多少个点。

| 应用 | 采样率 | 每秒点数 |
|---|---|---|
| 电话 | 8000 Hz | 8,000 |
| Whisper 输入 | 16000 Hz | 16,000 |
| CD 音质 | 44100 Hz | 44,100 |

**核心公式**：`N = round(duration × sample_rate)` 个采样点

## ✏️ 任务 0：先手算 N，再实现 `samples_count`

**第一步：手算下表**（不要运行代码，用 `N = round(duration × sr)` 计算）

| duration (s) | sample_rate (Hz) | N（手算）|
|---|---|---|
| 0.5 | 8 | ✏️ ? |
| 1.0 | 16000 | ✏️ ? |
| 0.25 | 44100 | ✏️ ? |

**第二步**：填完表后再往下实现函数。

In [ ]:
def samples_count(duration, sample_rate):
    """返回给定时长和采样率下的采样点总数 N。"""
    # ✏️ TODO: 返回 round(duration * sample_rate)
    return None  # ← 改这里


In [ ]:
# 用代码对答案——和手算结果一致吗？
assert samples_count(0.5, 8)     == 4,     f'期望 4，得到 {samples_count(0.5, 8)}'
assert samples_count(1.0, 16000) == 16000, f'期望 16000，得到 {samples_count(1.0, 16000)}'
assert samples_count(0.25, 44100)== round(0.25 * 44100)
assert samples_count(1.0, 10)    == 10

# 打印对照表
print(f'  duration=0.5s,  sr=8     → N={samples_count(0.5, 8)}')
print(f'  duration=1.0s,  sr=16000 → N={samples_count(1.0, 16000)}')
print(f'  duration=0.25s, sr=44100 → N={samples_count(0.25, 44100)}')
print('samples_count  OK')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 用 8 Hz 采样率采 0.5 秒：只有 4 个点，可以逐个检查
sr, duration = 8, 0.5
N = samples_count(duration, sr)
t = np.arange(N) / sr
x = np.sin(2 * np.pi * 1.0 * t)

print(f'N = {N} 个点（{duration}s × {sr}Hz）')
print(f't = {t}  （秒）')
print(f'x = {np.round(x, 3)}  （采样值）')

plt.stem(t, x, markerfmt='C0o', linefmt='C0-', basefmt='k-')
plt.xlabel('时间 (s)'); plt.ylabel('振幅')
plt.title(f'8 Hz 采样率下的 1 Hz 正弦波（N={N} 点）')
plt.tight_layout(); plt.show()

## ✏️ 任务 1：实现 `make_time_axis(duration, sample_rate)`

**第一步：手算**（duration=0.5, sr=8）

| 步骤 | 公式 | 结果 |
|---|---|---|
| 1. 算 N | `samples_count(0.5, 8)` | ✏️ ? |
| 2. 生成编号 n | `np.arange(N)` | ✏️ [?, ?, ?, ?] |
| 3. 转为时间 t | `n / sample_rate` | ✏️ [?, ?, ?, ?] |

**推理路线**（写完手算再看这里）：
1. N = `samples_count(duration, sample_rate)`
2. 编号 n = `np.arange(N)` → `[0, 1, 2, ..., N-1]`
3. 时间轴 t = n / sample_rate

返回：dtype=float64 的 numpy 数组，第一个元素为 0，最后一个 < duration。

In [ ]:
def make_time_axis(duration, sample_rate):
    # ✏️ TODO: 使用 samples_count + np.arange 实现
    return None  # ← 改这里


In [ ]:
import numpy as np

t = make_time_axis(0.5, 8)
assert isinstance(t, np.ndarray),          't 必须是 numpy 数组'
assert t.shape == (4,),                    f'shape 应为 (4,)，得到 {t.shape}'
assert np.allclose(t, [0.0, 0.125, 0.25, 0.375]), f'值不对：{t}'

t2 = make_time_axis(1.0, 16000)
assert t2.shape == (16000,)
assert abs(t2[0]) < 1e-12,   '第一个时刻应为 0'
assert t2[-1] < 1.0,         '最后一个时刻应 < duration（不含终点）'

print('make_time_axis  OK')
print(f'  t[:4] = {t}')

## 插曲：数组形状（shape）

NumPy 数组有**形状**（`shape`），描述它有几个维度、每个维度多长：

| 对象 | 例子 | shape | 用途 |
|---|---|---|---|
| 标量 | `3.14` | `()` | 单个数 |
| 1D 数组（向量）| `[0., 0.5, 1.]` | `(3,)` | 时间轴、音频信号 |
| 2D 数组（矩阵）| `[[1,2],[3,4]]` | `(2, 2)` | 频谱图、权重矩阵 |

本课所有信号都是 **1D 数组**（shape = `(N,)`）。
Module 3 线性代数和 Module 6 深度学习会大量使用 2D 矩阵。

In [ ]:
import numpy as np

scalar = np.float64(3.14)
vec    = np.array([0., 0.5, 1.])
mat    = np.array([[1, 2], [3, 4]])

for name, arr in [('标量', scalar), ('向量', vec), ('矩阵', mat)]:
    print(f'  {name:4s}  shape={str(arr.shape):10s}  ndim={arr.ndim}  dtype={arr.dtype}')

# 音频信号的形状
t = make_time_axis(1.0, 16000)
print(f'\n  音频时间轴  shape={t.shape}  → 一维，{len(t)} 个点')

## ✏️ 任务 2：实现 `make_sine(duration, sample_rate, frequency, amplitude=1.0)`

**第一步：手算**（duration=0.5s, sr=8Hz, freq=1Hz, A=1）

| n | t = n/sr (s) | θ = 2π·f·t (rad) | sin(θ)（精确到小数点后 2 位）|
|---|---|---|---|
| 0 | 0.000 | 0.000 | ✏️ ? |
| 1 | 0.125 | 0.785 | ✏️ ? |
| 2 | 0.250 | 1.571 | ✏️ ? |
| 3 | 0.375 | 2.356 | ✏️ ? |

**公式**：`x(t) = amplitude × sin(2π × frequency × t)`

**推理路线**：
1. 用 `make_time_axis` 生成时间轴 t
2. 返回 `amplitude × np.sin(2 × np.pi × frequency × t)`

In [ ]:
def make_sine(duration, sample_rate, frequency, amplitude=1.0):
    # ✏️ TODO: 使用 make_time_axis 和 np.sin 实现
    return None  # ← 改这里


In [ ]:
import numpy as np

# 基本验证
x = make_sine(1.0, 16, 2.0)
assert isinstance(x, np.ndarray)
assert x.shape == (16,)
assert abs(x.max() - 1.0) < 1e-9,   f'max 应为 1.0，得到 {x.max()}'
assert abs(x.min() + 1.0) < 1e-9,   f'min 应为 -1.0，得到 {x.min()}'

# 振幅验证
x2 = make_sine(1.0, 16, 2.0, amplitude=3.0)
assert abs(x2.max() - 3.0) < 1e-9,  f'amplitude=3 时 max 应为 3.0'

# 与 numpy 参考一致
t = make_time_axis(1.0, 16)
ref = np.sin(2 * np.pi * 2.0 * t)
assert np.allclose(x, ref), '结果与 np.sin 参考不一致'

# 对照手算：duration=0.5, sr=8, freq=1
x_check = make_sine(0.5, 8, 1.0)
print('手算对照（freq=1Hz, sr=8Hz, duration=0.5s）：')
print(f'  代码输出 = {np.round(x_check, 3)}')
print(f'  参考值   = {np.round(np.sin([0, np.pi/4, np.pi/2, 3*np.pi/4]), 3)}')
print('make_sine  OK')

In [ ]:
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
t = make_time_axis(dur, sr)

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, freq, color in zip(axes, [2, 5, 10], ['C0', 'C1', 'C2']):
    ax.plot(t, make_sine(dur, sr, freq), color=color)
    ax.set_ylabel(f'{freq} Hz'); ax.set_ylim(-1.3, 1.3); ax.grid(alpha=0.3)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('频率越高，单位时间内波峰越多')
plt.tight_layout(); plt.show()

try:
    from IPython.display import Audio, display
    x_audio = make_sine(1.5, 16000, 440)
    print('播放 440 Hz（A4 音）——你将听到标准「啦」音')
    display(Audio(x_audio, rate=16000))
except Exception:
    print('（非 Jupyter 环境，跳过播放）')

## 参数实验：频率与音高（pitch）

**预测问题（先回答，再运行验证）**：
- `make_sine(1.0, 16000, 880)` 的最大值是多少？
- 和 `make_sine(1.0, 16000, 440)` 的 shape 一样吗？
- 哪个频率听起来音调更高？

音高由频率决定。频率加倍 = 升高一个八度：

| 音名 | 频率 |
|---|---|
| A3（低八度）| 220 Hz |
| A4（标准「啦」）| 440 Hz |
| A5（高八度）| 880 Hz |

In [ ]:
try:
    from IPython.display import Audio, display
    sr = 16000
    for name, freq in [('A3 (220 Hz)', 220), ('A4 (440 Hz)', 440), ('A5 (880 Hz)', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')
        display(Audio(wave, rate=sr))
except Exception:
    sr = 16000
    for name, freq in [('A3', 220), ('A4', 440), ('A5', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')

## ✏️ 任务 3：实现 `signal_summary(x)`

返回字典，包含信号的统计摘要：
- `length`：数组长度（整数）
- `shape`：数组形状（元组）
- `max_abs`：最大绝对值（浮点数）
- `mean`：均值（浮点数）

**手算**（x = make_sine(1.0, 4, 1.0)，即 4 个点：[0, 1, 0, -1]）：

| 键 | 手算 |
|---|---|
| length | ✏️ ? |
| shape | ✏️ ? |
| max_abs | ✏️ ? |
| mean | ✏️ ?（4 个数的平均值）|

In [ ]:
def signal_summary(x):
    # ✏️ TODO: 返回包含 length / shape / max_abs / mean 的字典
    return {}  # ← 改这里


In [ ]:
import numpy as np

x_test = make_sine(1.0, 64, 3.0)
s = signal_summary(x_test)

assert s.get('length') == 64,              f"length 应为 64，得到 {s.get('length')}"
assert s.get('shape') == (64,),            f"shape 应为 (64,)，得到 {s.get('shape')}"
assert abs(s.get('max_abs', 0) - 1.0) < 0.02, f"max_abs 应约为 1.0"
assert abs(s.get('mean', 1)) < 0.05,       f"均值应接近 0"

# 手算验证：[0, 1, 0, -1] 的 4 个点
x4 = make_sine(1.0, 4, 1.0)
s4 = signal_summary(x4)
print('手算对照（4 个点 [0, 1, 0, -1]）：')
print(f'  x4 = {np.round(x4, 3)}')
print(f'  summary = {s4}')
print()
print('signal_summary  OK')
print(s)

## 本课收束

| 函数 | 作用 | 后续出现 |
|---|---|---|
| `samples_count` | N = duration × sr | 每个信号函数的第一步 |
| `make_time_axis` | 生成时间轴 | L32 numpy_signals, L33 sine_wave |
| `make_sine` | 生成正弦波 | L34 aliasing, L36 windows, L37 DFT |
| `signal_summary` | 汇总统计量 | L40 spectrum, L51 real_audio |

**本课用到的方法论**：先手算推导表，再写代码对答案。
这个方法贯穿整个 Aurora 课程——FFT 的每一步也会先手算再实现。

**核心直觉**：声音 = 一维 float64 数组，长度 = duration × sample_rate。
FFT 把这个数组变换到频域（frequency domain）——那是 L37 以后的事。

**下一课 L03**：谱图直觉——在学 FFT 之前先看结果。
你将看到上面各类声音的时频图，为 L37-L41 种下视觉直觉。